## Zelf een NN maken zonder library

We gaan nu zelf een Neuraal NEtwerk implementeren zonder gebruik te maken van libraries voor Neurale Netwerken. Sklearn gebruiken om de mnsit data te laden en numpy gebruiken mag uiteraard wel.

### Data laden

Laad de mnist data zoals je het eerder hebt geladen. Split dedara op in een test/training set. Bepaal zelf op hoeveel je split.

Normaliseer de data op 0-1 ipv de pixelwaardes. Dit heb je ook al eerder gedaan. 

Doe een one hot encoding op de labels van de data.

In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from sklearn.preprocessing import OneHotEncoder

# sparse_output=False: resultaat na encoding is normale NumPy array
encoder = OneHotEncoder(sparse_output=False)

# Load mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()


# Normalize images
train_images = train_images.astype(np.float32) / 255.0
test_images = test_images.astype(np.float32) / 255.0

train_images_flat = train_images.reshape(-1, 784)
test_images_flat = test_images.reshape(-1, 784)

# One hot encode labels
train_labels_oh = encoder.fit_transform(train_labels.reshape(-1, 1))
test_labels_oh = encoder.fit_transform(test_labels.reshape(-1, 1))


In [2]:
print(train_images_flat.shape)
print(test_images_flat.shape)

(60000, 784)
(10000, 784)


### Activatie functies

Maak de volgende activatiefuncties: 

    def relu(waardes)

en 

    def softmax(waardes) 

Als het goed is kun je ze uit de vorige opdracht gebruiken. 

In [3]:
def relu(arr):
    return np.maximum(0, arr)

def softmax(arr):
    arr = np.array(arr, dtype=np.float64)
    shifted = arr - np.max(arr, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)

testdata = np.array([-2.0, -0.5, 0.0, 1.5, 3.0])
batch = np.array([
    [1.0, 2.0, 3.0],
    [1.0, 0.0, -1.0]
])


print(relu(testdata))
print(softmax(batch))

print("Sum of softmax outputs for each row:")
for i in range(len(batch)):
    print(f"Row {i}: {sum(softmax(batch)[i])}")


[0.  0.  0.  1.5 3. ]
[[0.09003057 0.24472847 0.66524096]
 [0.66524096 0.24472847 0.09003057]]
Sum of softmax outputs for each row:
Row 0: 0.9999999999999999
Row 1: 0.9999999999999999


### Cross entropy loss

Maak een functie

    def cross_entropy(y_true, y_predicted)

functie om de grootte van loss te kunnen berekenen. We gaan de waarde van de loss gebruiken als validatie output. Het wordt dus aan het einde geprint.

In [4]:
y_true = np.array([[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]])
y_pred = np.array([[0.01, 0.02, 0.01, 0.05, 0.03, 0.02, 0.04, 0.75, 0.05, 0.02]])

def cross_entropy(y_true, y_predicted):
    e = 1e-9 # kleine waarde om log(0) te voorkomen
    return -np.mean(np.sum(y_true * np.log(y_predicted + e), axis=1))


ce_1 = cross_entropy(y_true, y_pred)
print(ce_1)


0.2876820711184476


### Toelichting op de functies

Alle  functies werken nu op een batch van plaatjes
niet op een los plaatje.

**relu(arr)**
- `arr`: NumPy-array van willekeurige vorm bijvoorbeeld de hidden pre-activatie met
  vorm (samples, hidden_nodes)
- Geeft voor elk element `max(0, x)` terug; negatieve waarden worden 0
- `np.maximum`. Zorgt er voor dat je het direct op de hele array kan uitvoeren (ipv elk los element)

**softmax(arr)**
- `arr`: 2D array met vorm (samples, klassen) bijvoorbeeld de output pre-activatie
  (logits) met vorm (samples, 10).
- Geeft per rij voor elk plaatje een kansverdeling terug die optelt tot 1.
- `axis=-1` zorgt dat de berekening per rij gebeurt. het aftrekken van het
  rij-maximum voorkomt overflow.

**cross_entropy(y_true, y_predicted)**
- `y_true`: 2D-array (samples, klassen) met de one-hot encoded labels.
- `y_predicted`: 2D-array (samples, klassen) met de voorspelde kansen uit softmax.
- Berekent per plaatje de categorische cross entropy en geeft het gemiddelde
  over alle plaatjes terug.
- `axis=1` sommeert per plaatje over de klassen; `np.mean` middelt over de plaatjes.
- De `+ 1e-9` voorkomt `log(0)`.

### Initialiseer je NN 

Maak de volgende variabelen:

- Kies hoeveel input nodes je hebt. 

784 input notes, een voor elke pixel van een mnist plaatje

- En hoeveel hidden layer nodes? 

Voor nu ga ik een hidden layer van 128 nodes aanhouden.

- En hoeveel output nodes? Leg uit waarom je dat hebt gekozen.

10 output nodes want je moet elk mogelijk getal coveren. 0-9 dus.

Maak voor nu 1 input layer en 1 hidden layer. JE mag dit later uitbreiden

Om de initiele gewichten een random waarde te geven kun je np.random.randn gebruiken. Deze geeft een normale verdeling met waarden die ongeveer liggen tussen -3 en +3.

Om grote getallen te voorkomen kun je deze weer kleiner maken door bijv *0.01 te doen, dan wordt de standaardafwijking 0.01

In [5]:
input_nodes_amount = 784
hidden_nodes_amount = 128
output_nodes_amount = 10

W1 = np.random.randn(input_nodes_amount, hidden_nodes_amount) * 0.01
b1 = np.zeros(hidden_nodes_amount)

W2 = np.random.randn(hidden_nodes_amount, output_nodes_amount) * 0.01
b2 = np.zeros(output_nodes_amount)

### Forward pass
Maak een forward propagation functie. We returnen ook tussenwaarden (cache) zodat we die voor backpropagation runnen gebruiken.

#### Stap 1

Maak functie def forward()
Als parameters hgebruik de volgende:
- inputdata X
- gewichtsmatrix W1 van de hidden layer nodes
- biasvector b1 van de hidden layer nodes
- gewichtsmatrix W2 van de output layer nodes
- biasvector b2 van de output layer nodes

Bereken eerst de invoer van de hidden layer. Vermenigvuldig de input met de gewichten en tel de bias erbij op.


Hint:

denk aan matrixvermenigvuldiging
de vorm moet kloppen: (samples, features) @ (features, hidden)

#### Stap 2

De uitkomst van stap 1 is de ruwe activatie van de hidden layer. Pas nu de door jou geiplementeerde ReLU-functie hierop toe.

#### Stap 3

Gebruik de output van de hidden layer als input voor de outputlaag. Vermenigvuldig met W2 en tel de bias b2 er bij op. Eigenlijk bijna hetzelfde als wat je in stap 1 hebt gedaan maar met andere dimensies.

#### Stap 4

De output van stap 3 zijn “scores” (logits), nog geen kansen. Zet deze scores om naar kansen met de softmax-functie die je eerder hebt gemaakt.

#### Stap 5


Voor de backpropagation stap heb je straks de tussenwaarden nodig.
Sla alle relevante variabelen op in één object (bijv. tuple)

- input
- hidden pre-activatie
- hidden activatie
- output pre-activatie
- voorspelling

#### Stap 6

return nu (bijvoorbeeld in een tuple vorm) de voorspelling en de object uit Stap 5

In [6]:
def forward(X, weight1, bias1, weight2, bias2):
    Z1 = X @ weight1 + bias1
    A1 = relu(Z1)
    Z2 = A1 @ weight2 + bias2
    A2 = softmax(Z2)
    cache = (X, Z1, A1, Z2, A2)
    return (A2, cache)

input_data = train_images_flat[:5]
actual_labels = train_labels_oh[:5]

pred, cache = forward(input_data, W1, b1, W2, b2)


print(actual_labels)
# print(pred.shape)
print(pred)
# print(cache)

[[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]
[[0.09987206 0.10058505 0.09933548 0.09974293 0.10011605 0.10046525
  0.10007749 0.10009225 0.10007764 0.09963579]
 [0.09995003 0.09907559 0.10037486 0.100896   0.1012475  0.09991703
  0.09901356 0.10056632 0.09983937 0.09911974]
 [0.10060198 0.09979611 0.09956679 0.1003783  0.09977072 0.10018761
  0.09932103 0.10027927 0.10003237 0.1000658 ]
 [0.09994411 0.09992326 0.09898334 0.10100057 0.10081493 0.1001655
  0.09926074 0.09996339 0.10009447 0.09984969]
 [0.10068621 0.0998664  0.09904025 0.10110303 0.1001308  0.09967297
  0.09931626 0.10015143 0.10032787 0.09970478]]


### Toelichting op de forward pass

De forward pass stuurt een batch plaatjes door het neurale netwerk en geeft een voorspelling terug

**forward(X, W1, b1, W2, b2)**
- `X`: 2D-array (samples, 784) met de plaatjes
- `W1, b1`: de gewichten en bias van de input naar de hidden layer
- `W2, b2`: de gewichten en bias van de hidden naar de output layer
- `Z1 = X @ W1 + b1`: de ruwe waardes van de hidden layer (input * gewichten, plus bias)
- `A1 = relu(Z1)`: de hidden-laag na ReLU (negatieve waardes worden 0)
- `Z2 = A1 @ W2 + b2`: de ruwe scores van de output layer (logits)
- `A2 = softmax(Z2)`: de logits omgezet naar kansen die per plaatje optellen tot 1. Dit is de voorspelling
- `cache`: bewaart `(X, Z1, A1, Z2, A2)` zodat de backward pass die tussenwaardes kan hergebruiken


### Backward pass

We gaan enkele functies voorbereiden die we in de les hebben gehad om de backpropagation stap uit te voeren.

We beginnen met berekening van de gradient van loss.

Tip: Je kunt de formule uit de slides van deze week gebruiken. 

Maak nu een methode 
        
        def compute_output_gradient(final_prediction, correct_answers)
        
die de output (laag) gradient teruggeeft. Je kunt de formule hierboven gebruiken.


In [7]:
def compute_output_gradient(final_prediction, correct_answers):
    N = final_prediction.shape[0] # aantal plaatjes pet batch
    # print(f"Batch size: {N}")
    return (final_prediction - correct_answers) / N

compute_output_gradient(pred, actual_labels)

array([[ 0.01997441,  0.02011701,  0.0198671 ,  0.01994859,  0.02002321,
        -0.17990695,  0.0200155 ,  0.02001845,  0.02001553,  0.01992716],
       [-0.18000999,  0.01981512,  0.02007497,  0.0201792 ,  0.0202495 ,
         0.01998341,  0.01980271,  0.02011326,  0.01996787,  0.01982395],
       [ 0.0201204 ,  0.01995922,  0.01991336,  0.02007566, -0.18004586,
         0.02003752,  0.01986421,  0.02005585,  0.02000647,  0.02001316],
       [ 0.01998882, -0.18001535,  0.01979667,  0.02020011,  0.02016299,
         0.0200331 ,  0.01985215,  0.01999268,  0.02001889,  0.01996994],
       [ 0.02013724,  0.01997328,  0.01980805,  0.02022061,  0.02002616,
         0.01993459,  0.01986325,  0.02003029,  0.02006557, -0.18005904]])

Maak nu methode 


    def compute_output_gradients(hidden_output, output_gradient)

die de gradienten van de output‑gewichten teruggeeft. Voeg aan de output van je methode ook de gradient van bias toe (als tuple bijv.). 

Tip: Je kunt de formule uit de slides van deze week gebruiken.


In [8]:
def compute_output_gradients(hidden_output, output_gradient):
    dW2 = hidden_output.T @ output_gradient
    db2 = np.sum(output_gradient, axis=0)
    return (dW2, db2)

dw2, db2 = compute_output_gradients(cache[2], compute_output_gradient(pred, actual_labels))

print("dW2 shape:", dw2.shape)
print("db2 shape:", db2.shape)

dW2 shape: (128, 10)
db2 shape: (10,)


Maak de methode 

    def compute_hidden_gradient(output_gradient, hidden_to_output_weights)
    
Deze zegt hoe sterk elke hidden neuron bijdraagt aan de fout.

Tip: Je kunt de formule uit de slides van deze week gebruiken.

In [9]:
def compute_hidden_gradient(output_gradient, hidden_to_output_weights):
    return output_gradient @ hidden_to_output_weights.T

Schrijf een python functie die afgeleide RELU berekent voor een getal:

    def relu_derivative(x)

Pas het aan zodat het voor een lijst van getallen werkt (eg met numpy)


In [10]:
def relu_derivative(x):
    return (x > 0).astype(float)

Maak nu methode 

    def compute_hidden_gradients(hidden_gradient, hidden_raw_gradient, input_data)

De functie moet de gradienten van de hidden-laag gewichten en bias teruggeven. Je hebt de afgeleide van de RELU nodig uit de vorige opgave, want alleen die gewichten doen mee.

Tip: Je kunt de formule uit de slides van deze week gebruiken.

In [11]:
def compute_hidden_gradients(hidden_gradient, hidden_raw_gradient, input_data):
    dZ1 = hidden_gradient * relu_derivative(hidden_raw_gradient)

    dW1 = input_data.T @ dZ1
    db1 = np.sum(dZ1, axis=0)
    return (dW1, db1)

Maak nu de backward propagation nmethode die de bovenstaatde methodes zal gbruiken.

    def backward(y_true, cache, W2):

De parameter cache zal uit de forward pass komen en als het goed is de volgende waardes bevatten:

- input
- hidden pre-activatie
- hidden activatie
- output pre-activatie
- voorspelling

Die kun je dus daaruit unpacken.

Doe nu de volgende stappen achter elkaar:

- Bereken de output gradient (gebruik je eerder gemaakte methode)
- Bereken de output gradients (voor gewichten en bias) (gebruik je eerder gemaakte methode en output gradient)

- Bereken de hidden gradient (gebruik je eerder gemaakte methode)
- Bereken de hidden gradients (voor gewichten en bias) (gebruik je eerder gemaakte methode en hidden gradient)

return de output gradients en de hidden gradients bijvoorbeeld in de vorm van een tuple:

    return dW1, db1, dW2, db2

In [12]:
def backward(y_true, cache, W2):
    # Tussenwaarden uit de forward pass uitpakken
    X, Z1, A1, Z2, A2 = cache

    # Fout bij de output: voorspelling - juiste antwoord (gemiddeld over de batch)
    output_gradient = compute_output_gradient(A2, y_true)

    # Gradient van output-gewichten W2 en bias b2
    dW2, db2 = compute_output_gradients(A1, output_gradient)

    # Foutsignaal terugsturen naar de hidden-laag via W2
    hidden_gradient = compute_hidden_gradient(output_gradient, W2)

    # Gradient van hidden-gewichten W1 en bias b1 (relu-afgeleide blokkeert inactieve nodes)
    dW1, db1 = compute_hidden_gradients(hidden_gradient, Z1, X)

    # Alle vier de gradienten teruggeven voor de update-stap
    return (dW1, db1, dW2, db2)

### Toelichting op de backward pass

De backward pass rekent vanaf de fout aan het eind terug naar voren, en bepaalt voor elk gewicht en elke bias hoe die moet veranderen. Een naam met een `d` ervoor (zoals `dW2`) is steeds "de aanpassing voor" en heeft dezelfde np shape als het origineel

**compute_output_gradient(final_prediction, correct_answers)**
- Berekent de fout bij de output: `voorspelling - juiste antwoord`.
- Delen door het aantal plaatjes geeft het gemiddelde over de batch.
- Een negatief getal betekent "deze kans moet omhoog", een positief getal "omlaag".

**compute_output_gradients(hidden_output, output_gradient)**
- Berekent `dW2` en `db2`: hoe de output-gewichten en bias moeten veranderen.
- `dW2 = hidden_output.T @ output_gradient`: combineert hoe actief elke hidden-node was met hoe fout de output was.
- `db2 = np.sum(output_gradient, axis=0)`: telt de fout op over alle plaatjes (1 value per output node).

**compute_hidden_gradient(output_gradient, hidden_to_output_weights)**
- Stuurt de fout terug naar de hidden layer via de gewichten `W2`.
- `output_gradient @ W2.T` geeft per hidden node hoeveel die aan de fout bijdroeg.

**relu_derivative(x)**
- De afgeleide van ReLU: `1` waar `x > 0`, anders `0`.
- `(x > 0).astype(float)` zet de True/False-waardes om naar 1.0/0.0.
- Wordt gebruikt om de fout te blokkeren bij nodes die "uit" stonden.

**compute_hidden_gradients(hidden_gradient, hidden_raw_gradient, input_data)**
- Berekent `dW1` en `db1`: hoe de hidden gewichten en bias moeten veranderen.
- `dZ1 = hidden_gradient * relu_derivative(hidden_raw_gradient)`: blokkeert de fout bij inactieve nodes.
- `dW1 = input_data.T @ dZ1`: combineert de input met het foutsignaal.
- `db1 = np.sum(dZ1, axis=0)`: telt op over alle plaatjes (1 value per hidden node).

**backward(y_true, cache, W2)**
- Rijgt de functies hierboven aan elkaar in de juiste volgorde.
- Pakt de tussenwaardes uit de cache en geeft `dW1, db1, dW2, db2` terug voor de update-stap.

### De training loop. 

JE mag het volgende stukje code gebruiken om je netwerk te trainen. MAar je mag uiteraard ook zelf een andere opzet maken.


In [13]:
lr = 0.1
batch_size = 128
n = train_images_flat.shape[0]
epochs = 20

print(f"Number of training samples: {n}")

for epoch in range(epochs):

    # train in batches van batch_size
    for start in range(0, n, batch_size):
        b = slice(start, start + batch_size)
        voorspelling, cache = forward(train_images_flat[b], W1, b1, W2, b2)
        dW1, db1, dW2, db2 = backward(train_labels_oh[b], cache, W2)
        W1 -= lr*dW1; b1 -= lr*db1; W2 -= lr*dW2; b2 -= lr*db2

    # einde epoch: loss/acc op de hele set
    pred_all, _ = forward(train_images_flat, W1, b1, W2, b2)
    loss = cross_entropy(train_labels_oh, pred_all)
    acc = np.mean(np.argmax(pred_all, axis=1) == train_labels)
    print(f"Epoch {epoch + 1}, Loss: {loss:.4f}, Acc: {acc:.3f}")

Number of training samples: 60000
Epoch 1, Loss: 0.3594, Acc: 0.893
Epoch 2, Loss: 0.2842, Acc: 0.917
Epoch 3, Loss: 0.2399, Acc: 0.930
Epoch 4, Loss: 0.2060, Acc: 0.940
Epoch 5, Loss: 0.1799, Acc: 0.948
Epoch 6, Loss: 0.1591, Acc: 0.954
Epoch 7, Loss: 0.1422, Acc: 0.959
Epoch 8, Loss: 0.1285, Acc: 0.964
Epoch 9, Loss: 0.1171, Acc: 0.967
Epoch 10, Loss: 0.1076, Acc: 0.970
Epoch 11, Loss: 0.0996, Acc: 0.972
Epoch 12, Loss: 0.0927, Acc: 0.974
Epoch 13, Loss: 0.0866, Acc: 0.976
Epoch 14, Loss: 0.0814, Acc: 0.978
Epoch 15, Loss: 0.0766, Acc: 0.979
Epoch 16, Loss: 0.0724, Acc: 0.980
Epoch 17, Loss: 0.0685, Acc: 0.981
Epoch 18, Loss: 0.0651, Acc: 0.982
Epoch 19, Loss: 0.0619, Acc: 0.983
Epoch 20, Loss: 0.0589, Acc: 0.984


In [14]:
test_pred, _ = forward(test_images_flat, W1, b1, W2, b2)

test_loss = cross_entropy(test_labels_oh, test_pred)
test_acc = np.mean(np.argmax(test_pred, axis=1) == test_labels)

print("Final evaluation:")
print(f"Test loss: {test_loss:.4f}, Test accuracy: {test_acc:.3f}")

Final evaluation:
Test loss: 0.0878, Test accuracy: 0.975


One hot encoding toepassen (op de labels)

### Experimenteer

Werkt het? Mooi, dan kun je nu zelf experimenteren met verschillende setup parameters en kijken wat het beste werkt.

In [ ]:
# Export weights
destination = input("Desitination path")
np.save(destination + "W1.npy", W1, True)
np.save(destination + "b1.npy", b1, True)
np.save(destination + "W2.npy", W2, True)
np.save(destination + "b2.npy", b2, True)